# German multimodal TikTok archive: analysis of the local CSV release

[Weizenbaum data record](https://www.weizenbaum-library.de/items/04482f4d-3f73-44e9-859b-f815b089caab) · [Data report](https://www.weizenbaum-library.de/bitstreams/54049d98-d6c8-4c66-a890-6360118c22f9/download). This source is a data report, not a peer-reviewed paper. Its reported collection used a German IP and a logged-out TikTok Explore page from 31 July to 4 August 2023. The four released CSVs are now stored locally. We distinguish **Explore appearances** from **unique videos**, **comment-count snapshots** from **individual comments**, and **embeddings** from raw media. Run this notebook from the repository root or its `Oct 9` folder.

## 1. Local files and schema

**Purpose:** Verify the four CSVs on disk and identify the metadata fields and analytical unit.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'downloaded_datasets' / 'german_multimodal'
names = ['meta_data.csv', 'text_embeddings.csv', 'audio_embeddings.csv', 'video_embeddings.csv']
files = pd.DataFrame([{'file': name, 'present': (DATA / name).is_file(),
                       'size_MB': round((DATA / name).stat().st_size / 1e6, 2) if (DATA / name).is_file() else None}
                      for name in names])
display(files)
missing = files.loc[~files.present, 'file'].tolist()
if missing:
    raise FileNotFoundError(f'Missing required local CSV files: {missing}')
metadata = pd.read_csv(DATA / 'meta_data.csv', dtype={'id_hash': str, 'author_hash': str}, low_memory=False)
print('Metadata columns:', metadata.columns.tolist())
print('Metadata rows:', len(metadata), 'distinct video hashes:', metadata.id_hash.nunique())
display(metadata.isna().sum().rename('missing_rows').to_frame())

,file,present,size_MB
0,meta_data.csv,True,5.49
1,text_embeddings.csv,True,27.97
2,audio_embeddings.csv,True,129.93
3,video_embeddings.csv,True,222.58


Metadata columns: ['author_followers', 'author_likes', 'author_videos', 'timestamp', 'unix_timestamp', 'is_duet', 'is_ad', 'likes', 'comments', 'shares', 'plays', 'hashtags', 'category', 'date_scraping', 'author_hash', 'music_id_hash', 'music_author_hash', 'music_name_hash', 'id_hash']
Metadata rows: 11242 distinct video hashes: 5137


,missing_rows
author_followers,0
author_likes,0
author_videos,0
timestamp,0
unix_timestamp,0
is_duet,0
is_ad,0
likes,0
comments,0
shares,0


**What this shows:** The metadata file contains engagement-count fields (`likes`, `comments`, `shares`, `plays`), categories, dates, hashtags and hashed IDs. It does not contain comment text or raw videos. Do not treat the 19 metadata columns as 19 independent variables with complete information: `hashtags` has missing values.

## 2. Paper/release count reconciliation

**Purpose:** Reproduce the source's 11,242 appearance rows and 5,137 unique-video count, and check repeated appearances.

In [2]:
overview = pd.DataFrame([{
    'source_reported_appearance_rows': 11242,
    'local_appearance_rows': len(metadata),
    'difference_in_appearance_rows': len(metadata) - 11242,
    'source_reported_unique_videos': 5137,
    'local_unique_video_hashes': metadata.id_hash.nunique(),
    'difference_in_unique_videos': metadata.id_hash.nunique() - 5137,
    'unique_creator_hashes_across_appearances': metadata.author_hash.nunique(),
    'exact_duplicate_rows': int(metadata.duplicated().sum()),
}])
display(overview)
appearances = metadata.groupby('id_hash').size()
display(appearances.describe(percentiles=[.25, .5, .75, .9, .99]).to_frame('appearances_per_video'))
print('Videos appearing more than once:', int(appearances.gt(1).sum()))

,source_reported_appearance_rows,local_appearance_rows,difference_in_appearance_rows,source_reported_unique_videos,local_unique_video_hashes,difference_in_unique_videos,unique_creator_hashes_across_appearances,exact_duplicate_rows
0,11242,11242,0,5137,5137,0,2992,0


,appearances_per_video
count,5137.000000
mean,2.188437
std,0.987026
min,1.000000
25%,1.000000
50%,2.000000
75%,3.000000
90%,4.000000
99%,4.640000
max,5.000000


Videos appearing more than once: 3676


**What this shows:** Both source-reported headline counts are exactly reproduced. A video appears 2.19 times on average; 3,676 video hashes occur more than once. Repeated appearances are not extra distinct videos or automatically duplicate data errors.

## 3. Publication and scraping dates

**Purpose:** Separate when a video was posted from when researchers captured its Explore-page appearance.

In [3]:
metadata['scrape_date'] = pd.to_datetime(metadata.date_scraping, format='%d_%m_%Y', errors='coerce')
metadata['posted_at'] = pd.to_datetime(metadata.timestamp, errors='coerce')
print('Unparseable scrape dates:', int(metadata.scrape_date.isna().sum()))
print('Unparseable posting timestamps:', int(metadata.posted_at.isna().sum()))
print('Observed post-publication range:', metadata.posted_at.min(), 'to', metadata.posted_at.max())
display(metadata.groupby('scrape_date').agg(
    appearance_rows=('id_hash', 'size'), unique_videos=('id_hash', 'nunique'),
    unique_creators=('author_hash', 'nunique')).reset_index())

Unparseable scrape dates: 0
Unparseable posting timestamps: 0
Observed post-publication range: 2023-04-10 08:38:09 to 2023-07-31 18:59:48


,scrape_date,appearance_rows,unique_videos,unique_creators
0,2023-07-31,2247,2247,1584
1,2023-08-01,2270,2270,1567
2,2023-08-02,2239,2239,1579
3,2023-08-03,2244,2244,1571
4,2023-08-04,2242,2242,1569


**What this shows:** The file covers five scraping days, 31 July–4 August 2023. The observed video publication timestamps range from 10 April to 31 July 2023. These are two different time windows.

## 4. Categories and repeat observations

**Purpose:** Describe which Explore categories were captured and whether videos cross categories.

In [4]:
display(metadata.category.value_counts(dropna=False).rename_axis('category').to_frame('appearance_rows'))
print('Number of distinct categories:', metadata.category.nunique())
print('Videos observed in more than one category:',
      int(metadata.groupby('id_hash').category.nunique().gt(1).sum()))
print('Video-date pairs with multiple appearance rows:',
      int(metadata.duplicated(['id_hash', 'scrape_date']).sum()))

,appearance_rows
category,
pets_nature,1029
society,1027
comedy_drame,1024
lifestyle,1023
relationships,1023
dance_music,1022
informative,1021
cars,1021
sport,1018


Number of distinct categories: 11
Videos observed in more than one category: 0
Video-date pairs with multiple appearance rows: 0


**What this shows:** There are 11 category labels with roughly 1,000 appearance rows each. In this release, a video hash is not observed under multiple category labels, and the same video is not repeated twice on a single scraping day. The labels `comedy_drame` and `entertaiment` are retained exactly as released.

## 5. One observation per video and comment-count distribution

**Purpose:** Avoid counting repeated daily snapshots as different videos; use the last available scrape for each video.

In [5]:
latest = metadata.sort_values(['scrape_date', 'id_hash']).drop_duplicates('id_hash', keep='last').copy()
assert latest.id_hash.is_unique
print('Unique videos in latest-snapshot table:', len(latest))
print('Unique creators in latest-snapshot table:', latest.author_hash.nunique())
print('Video hashes associated with more than one creator hash:',
      int(metadata.groupby('id_hash').author_hash.nunique().gt(1).sum()))
print('Videos whose recorded comment count changed between observed days:',
      int(metadata.groupby('id_hash').comments.nunique().gt(1).sum()))
display(latest[['comments', 'likes', 'shares', 'plays']]
        .describe(percentiles=[.25, .5, .75, .9, .99]).round(2))
print('Videos with zero recorded comments:', int(latest.comments.eq(0).sum()))

Unique videos in latest-snapshot table: 5137
Unique creators in latest-snapshot table: 2986
Video hashes associated with more than one creator hash: 6
Videos whose recorded comment count changed between observed days: 2698


,comments,likes,shares,plays
count,5137.00,5137.00,5137.00,5.137000e+03
mean,3206.64,415754.78,7517.46,5.629729e+06
std,8100.18,815946.98,21781.24,1.151130e+07
min,0.00,0.00,0.00,0.000000e+00
25%,413.00,59300.00,469.00,9.513000e+05
50%,1254.00,159900.00,1877.00,2.300000e+06
75%,3250.00,399500.00,6302.00,5.800000e+06
90%,7374.80,1000000.00,17700.00,1.360000e+07
99%,30664.00,3864000.00,85876.00,4.758400e+07
max,312200.00,15000000.00,629300.00,2.499000e+08


Videos with zero recorded comments: 53


**What this shows:** For the 5,137 unique videos, the last observed comment-count snapshot has a mean of about 3,187 and a median of about 1,254. These are **counts of comments attached to videos**, not rows of comment text. 2,698 videos have different comment counts on different observed days, so the choice of snapshot matters. The Explore sample strongly favors visible/popular material and should not be generalized to all TikTok videos.

## 6. Comments by category

**Purpose:** Find descriptive differences across Explore categories using one snapshot per video.

In [6]:
by_category = latest.groupby('category').comments.agg(
    videos='count', mean_comment_count='mean', median_comment_count='median',
    zero_comment_share_percent=lambda s: 100 * s.eq(0).mean())
display(by_category.sort_values('median_comment_count', ascending=False).round(2))
print('Spearman rank correlation: comment count versus play count:',
      round(latest[['comments', 'plays']].corr(method='spearman').iloc[0, 1], 3))

,videos,mean_comment_count,median_comment_count,zero_comment_share_percent
category,,,,
entertaiment,455,7046.04,4339.0,0.66
comedy_drame,453,4287.05,2273.0,0.22
society,462,3374.67,1889.5,1.30
sport,456,4087.43,1685.0,1.75
lifestyle,468,3701.31,1645.0,1.07
fashion,464,3021.54,1419.0,0.86
dance_music,483,2561.33,1152.0,2.07
pets_nature,474,3091.00,730.0,0.63
informative,473,1863.51,413.0,0.42


Spearman rank correlation: comment count versus play count: 0.682


**What this shows:** The `entertaiment` category has the highest median comment count (4,339); `cars` has the lowest (327). The comment–play Spearman correlation is about 0.682. These are associations in a selected Explore-page sample, not evidence that a category causes comments or that comment text is available.

## 7. Creator concentration

**Purpose:** Check how many distinct sampled videos are associated with each hashed creator.

In [7]:
per_creator = latest.groupby('author_hash').size()
display(per_creator.describe(percentiles=[.25, .5, .75, .9, .99]).to_frame('unique_sampled_videos_per_creator'))
print('Creators with exactly one sampled video:', int(per_creator.eq(1).sum()))

,unique_sampled_videos_per_creator
count,2986.000000
mean,1.720362
std,2.584161
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
90%,3.000000
99%,10.150000
max,52.000000


Creators with exactly one sampled video: 2235


**What this shows:** The one-video-per-creator median reflects this Explore sample, **not** a creator's complete posting history. The full appearance table has 2,992 creator hashes; the latest one-row-per-video table has 2,986. Six video hashes have different creator hashes across appearances. This is a data-quality caveat, not a reason to silently rewrite IDs.

## 8. Text, audio and video embedding coverage

**Purpose:** Verify the dimensions, uniqueness, and metadata overlap of each derived modality without loading hundreds of megabytes of vectors.

In [8]:
embedding_rows = []
metadata_ids = set(metadata.id_hash)
for name in ['text_embeddings.csv', 'audio_embeddings.csv', 'video_embeddings.csv']:
    path = DATA / name
    header = pd.read_csv(path, nrows=0).columns.tolist()
    hashes = pd.read_csv(path, usecols=['id_hash'], dtype=str).id_hash
    embedding_rows.append({
        'file': name, 'rows': len(hashes), 'unique_video_hashes': hashes.nunique(),
        'numeric_embedding_dimensions': len(header) - 1,
        'repeated_video_hash_rows': int(hashes.duplicated().sum()),
        'overlap_with_metadata_videos': len(set(hashes) & metadata_ids),
        'coverage_of_metadata_videos_percent': 100 * len(set(hashes) & metadata_ids) / len(metadata_ids),
    })
display(pd.DataFrame(embedding_rows).round(2))

,file,rows,unique_video_hashes,numeric_embedding_dimensions,repeated_video_hash_rows,overlap_with_metadata_videos,coverage_of_metadata_videos_percent
0,text_embeddings.csv,5137,5137,512,0,5137,100.0
1,audio_embeddings.csv,5137,5137,4096,0,5137,100.0
2,video_embeddings.csv,5137,5137,2304,0,5137,100.0


**What this shows:** All three embedding files contain exactly one row for each of the 5,137 hashed videos and join to the metadata at 100% by `id_hash`. Their dimensions are 512 (text), 4,096 (audio), and 2,304 (video). These vectors are derived numeric features, not descriptions, comment text, audio files, or playable videos.

## Findings and limits

- **Locally verified:** 11,242 appearance rows, 5,137 unique video hashes, 11 category labels, and 2,992 distinct creator hashes across appearance rows. The first two counts match the data report.
- **Repeated observations:** 3,676 videos appear on more than one scraping day. Comment counts change across days for 2,698 videos. For video-level summaries, this notebook uses each video's **last observed scrape**, yielding 5,137 rows.
- **Comments:** the last-snapshot mean is about **3,187** comments per video and the median is about **1,254**. These are metadata counts; there are **no individual comment rows**. Category comparisons and correlations are descriptive only.
- **Multimodal data:** text/audio/video embeddings cover all 5,137 hashed videos, with 512/4,096/2,304 numeric dimensions respectively. They cannot reconstruct or substitute for raw media.
- **Limits:** the collection is a logged-out German Explore-page sample, not all TikTok. Video URLs, descriptions, and unhashed IDs are not released. Six creator hashes present in the appearance table do not remain in the last-snapshot table; inspect such inconsistencies before creator-level linkage.